# Relay legalize

Legalize pass 用于将高级算子转换为目标设备支持的形式，或替换为等效的操作序列。
本测试文件涵盖了多种 legalize 场景，包括算子替换、无操作处理、多算子替换、多输入处理
以及特定硬件目标下的卷积操作 legalize。

In [1]:
import set_env

In [2]:
import numpy as np
import pytest
import tvm
from tvm import te

from tvm import relay
from tvm.contrib import graph_executor
from tvm.relay import transform, analysis
from tvm.relay.testing.temp_op_attr import TempOpAttr

def run_opt_pass(expr, passes):
    """
    执行优化 passes 的通用函数。
    
    参数:
    expr: 要优化的 Relay 表达式或函数
    passes: 单个优化 pass 或优化 pass 列表
    
    返回值:
    优化后的 Relay 表达式或函数
    """
    passes = passes if isinstance(passes, list) else [passes]
    mod = tvm.IRModule.from_expr(expr)
    print("变换前后：")
    mod.show()
    seq = tvm.transform.Sequential(passes)
    with tvm.transform.PassContext(opt_level=3):
        mod = seq(mod)
    entry = mod["main"]
    mod.show()
    return entry if isinstance(expr, relay.Function) else entry.body

## 算子替换

In [3]:
def before():
    x = relay.var("x", shape=(1, 64, 56, 56))
    weight = relay.var("weight", shape=(64, 64, 3, 3))
    y = relay.nn.conv2d(x, weight, channels=64, kernel_size=(3, 3), padding=(1, 1))
    y = relay.nn.relu(y)
    y = relay.Function([x, weight], y)
    return y

In [4]:
def legalize_conv2d(attrs, inputs, types):
    """conv2d 算子的 legalize 函数，将权重乘以 2.0。"""
    data, weight = inputs
    weight = relay.multiply(weight, relay.const(2.0, "float32"))
    return relay.nn.conv2d(data, weight, **attrs)

In [5]:
# 临时设置 conv2d 的 legalize 函数
with TempOpAttr("nn.conv2d", "FTVMLegalize", legalize_conv2d):
    a = before()
    a = run_opt_pass(a, transform.Legalize())

变换前后：


## 测试通过返回 `None` 不做任何操作的情况

In [7]:
def before():
    """定义 legalize 前的原始计算图。"""
    x = relay.var("x", shape=(1, 64, 56, 56))
    y = relay.nn.global_max_pool2d(x)
    y = relay.Function([x], y)
    return y

called = [False]

def legalize_conv2d(attrs, inputs, types):
    """global_max_pool2d 操作的 legalize 函数，仅标记被调用，返回 None。"""
    called[0] = True
    return None

# 临时设置 global_max_pool2d 的 legalize 函数
with TempOpAttr("nn.global_max_pool2d", "FTVMLegalize", legalize_conv2d):
    a = before()
    a = run_opt_pass(a, transform.Legalize())
    b = run_opt_pass(before(), transform.InferType())

# 验证结果与原始图相同，但 legalize 函数确实被调用了
tvm.ir.assert_structural_equal(a, b)
assert called[0]

变换前后：


变换前后：


## 测试同时替换多个算子

In [8]:
def before():
    """定义 legalize 前的原始计算图。"""
    x = relay.var("x", shape=(1, 64, 56, 56))
    weight = relay.var("weight", shape=(64, 64, 3, 3))
    y = relay.nn.conv2d(x, weight, channels=64, kernel_size=(3, 3), padding=(1, 1))
    y = relay.nn.relu(y)
    y = relay.Function([x, weight], y)
    return y

def legalize_conv2d(attrs, inputs, types):
    """conv2d 操作的 legalize 函数，将权重乘以 2.0。"""
    data, weight = inputs
    weight = relay.multiply(weight, relay.const(2.0, "float32"))
    return relay.nn.conv2d(data, weight, **attrs)

def legalize_relu(attrs, inputs, types):
    """relu 操作的 legalize 函数，在 relu 前添加一个与 0 的加法操作。"""
    data = inputs[0]
    add = relay.add(tvm.relay.const(0, "float32"), data)
    return relay.nn.relu(add)

In [9]:
# 临时设置 conv2d 和 relu 的 legalize 函数
with TempOpAttr("nn.conv2d", "FTVMLegalize", legalize_conv2d):
    with TempOpAttr("nn.relu", "FTVMLegalize", legalize_relu):
        a = before()
        a = run_opt_pass(a, transform.Legalize())

变换前后：


## 测试处理多个输入的算子的 legalize 功能

In [10]:
def before():
    """定义 legalize 前的原始计算图，包含多输入的 concatenate 操作。"""
    x = relay.var("x", shape=(1, 64, 56, 56))
    y = relay.var("y", shape=(1, 64, 56, 20))
    z = relay.var("z", shape=(1, 64, 56, 10))
    func = relay.concatenate([x, y, z], axis=3)
    func = relay.Function([x, y, z], func)
    return func

def legalize_concatenate(attrs, inputs, types):
    """concatenate 操作的 legalize 函数，仅验证输入类型和结构。"""
    # 验证多输入情况的正确处理
    assert len(inputs) == 1
    assert isinstance(inputs[0], tvm.relay.expr.Tuple)
    assert len(types) == 2
    assert isinstance(types[0], tvm.relay.ty.TupleType)
    assert isinstance(types[1], tvm.relay.ty.TensorType)
    return None

In [12]:
# 临时设置 concatenate 的 legalize 函数
with TempOpAttr("concatenate", "FTVMLegalize", legalize_concatenate):
    a = before()
    a = run_opt_pass(a, transform.Legalize())

变换前后：


## 测试在不同 ARM CPU 目标下的卷积 2D NHWC 布局的 legalize 功能

In [13]:
for target, exp_in_channels in [
        (
            "llvm -device=arm_cpu -mtriple=aarch64-linux-gnu",
            8,
        ),
        (
            "llvm --device=arm_cpu --mtriple=aarch64-linux-gnu -mattr=+v8.2a,+dotprod",
            3,
        ),
        (
            "llvm --device=arm_cpu --mtriple=aarch64-linux-gnu -mattr=+v8.2a,+i8mm",
            8,
        ),
        (
            "llvm -device=arm_cpu -mtriple=aarch64-linux-gnu -mattr=+neon",
            8,
        ),
        (
            "llvm -device=arm_cpu -mtriple=armv8l-linux-gnu -mattr=+neon",
            8,
        ),
    ]:
    target = tvm.target.Target(target)

    dtype = "int8"
    data_layout = "NHWC"
    kernel_layout = "HWIO"
    in_channels = 3
    out_channels = 4
    kernel_size = (1, 1)

    x = relay.var("x", shape=(1, 1, 1, in_channels), dtype=dtype)
    weight = relay.var("weight", shape=(1, 1, in_channels, out_channels), dtype=dtype)
    out = relay.nn.conv2d(
        x,
        weight,
        kernel_size=kernel_size,
        channels=out_channels,
        data_layout=data_layout,
        kernel_layout=kernel_layout,
        out_dtype=dtype,
    )

    # 在指定目标下执行 legalize pass
    with target:
        out = run_opt_pass(out, transform.Legalize())

    # 获取 legalize 后的实际输入通道数
    act_in_channels = out.args[0].type_args[0].shape[3]

    # 验证输入通道数是否符合预期（不同的 ARM CPU 特性可能需要不同的输入通道数对齐）
    assert act_in_channels == exp_in_channels, "Actual input channels = " + str(act_in_channels)

变换前后：


变换前后：


变换前后：


变换前后：


变换前后：
